# Part1：LLMの機能を試す

## 事故画像（PNG）の解析
事故の画像を、AI_COMPLETE関数を使って解析します。

In [ ]:
## 画像を表示
from snowflake.snowpark.context import get_active_session
from IPython.display import Image, display

session = get_active_session()
image_bytes = session.file.get_stream(
    '@handson.car_insurance.image/accident_image.png',
    decompress=False
).read()
display(Image(data=image_bytes))

In [ ]:
%%sql -r dataframe_1
-- 関数実行
SELECT AI_COMPLETE(
  'pixtral-large',
  PROMPT(
    'この画像{0}の内容を日本語で詳しく解説してください。',
    TO_FILE('@handson.car_insurance.image', 'accident_image.png')
  )
) AS image_description;

In [ ]:
## 見やすいように出力
import json

raw = dataframe_1.iloc[0, 0]
text = json.loads(raw) if raw.startswith('"') else raw
print(text)

## 保険金請求書（PDF）からテキストを抽出し、構造化データへ変換
- AI_EXTRACT関数を使い、PDFから必要な情報を抽出
- 抽出したデータをテーブルへ格納（構造化データ変換）

In [ ]:
%%sql -r dataframe_3
-- テキスト抽出
CREATE OR REPLACE TABLE handson.car_insurance.extract_table AS
SELECT
  relative_path AS file_name,
  AI_EXTRACT(
    file => TO_FILE('@handson.car_insurance.pdf', relative_path),
    responseFormat => [
      ['name', '保険金請求者の氏名は何ですか？'],
      ['phone', '日中連絡先は何ですか？'],
      ['security_number', '証券番号は何ですか？'],
      ['accident_situation', '発生状況は何ですか？'],
      ['billing_date',  '請求日はいつですか？']
    ]
  ) AS document_extracts
FROM DIRECTORY(@handson.car_insurance.pdf)
WHERE relative_path = 'hoken_seikyu.pdf';

In [ ]:
%%sql -r dataframe_5
-- 抽出したテキストから指定した項目を列としてテーブル形式で格納
CREATE OR REPLACE TABLE handson.car_insurance.car_claim_form AS
WITH long AS (
  SELECT
      t.file_name,
      f.value::string AS field_name,
      NULLIF(GET(t.document_extracts:response, f.value)::string, 'None') AS field_value
  FROM handson.car_insurance.extract_table t,
       LATERAL FLATTEN(input => OBJECT_KEYS(t.document_extracts:response)) f
)
SELECT *
FROM long
PIVOT (
  MAX(field_value) FOR field_name
  IN ('name', 'phone', 'security_number', 'accident_situation', 'billing_date')
) AS p (file_name, name, phone, security_number, accident_situation, billing_date);

In [ ]:
%%sql -r dataframe_6
-- 確認
SELECT * FROM handson.car_insurance.car_claim_form;

## 問い合わせ内容に対して、タグ付け
問い合わせ内容（テキスト）を指定したカテゴリに分類します。

In [ ]:
%%sql -r dataframe_7
-- 関数を適用
SELECT 
    AI_CLASSIFY(
        '高速道路で前の車に追突してしまいました。相手にケガはありませんが車両に損傷があります。',  -- 分類するテキスト
        ['事故報告', '保険料問い合わせ', '契約手続き']  -- 分類カテゴリのリスト
    ) as classification;

## PIIを含むデータのマスキング（多層マスキングパイプライン）
- 署名を削除
- 正規表現で対応できるものをマスキング
- 残ったものをAI_REDACT関数でマスキング

In [ ]:
%%sql -r dataframe_8
-- 実行
WITH raw_text AS (
  SELECT '以下は、お客様からの申請情報です。

氏名：山田太郎
年齢と性別：38歳の男性
生年月日：1985年12月15日
住所：〒150-0001 東京都渋谷区神宮前1-2-3
電話番号：03-1234-5678
携帯：090-1111-2222
メールアドレス：taro.yamada@example.co.jp
マイナンバー：123456789012
証券番号：1234567890
パスポート番号：TK1234567
クレジットカード番号：4532-1234-5678-9010、有効期限：12/25、CVV：123

ご確認のほど、よろしくお願いします。
森
' AS text
),
step1_remove_signature AS (
  SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'llama3.1-8b',
    '以下のテキストから末尾の署名部分（名前だけの行）を除去して、本文だけをそのまま返してください。数字・記号・メールアドレスなど本文の内容は一文字たりとも変更しないでください。追加の説明も不要です:\n' || text
  ) AS text
  FROM raw_text
),
step2_regex_mask AS (
  SELECT REGEXP_REPLACE(
    REGEXP_REPLACE(
      text,
      'マイナンバー：[0-9]{12}',
      'マイナンバー：[MY_NUMBER]'
    ),
    '証券番号：[0-9]{7,10}',
    '証券番号：[SECURITIES_NUMBER]'
  ) AS text
  FROM step1_remove_signature
)
SELECT AI_REDACT(text) AS comprehensive_test_with_llm
FROM step2_regex_mask;

In [ ]:
## 見やすいように出力
import json

raw = dataframe_8.iloc[0, 0]
text = json.loads(raw) if raw.startswith('"') else raw
print(text)

## 事故対応に対する顧客FBデータの集計
AI_AGGによるフィードバックデータから、サービス全体の満足度を要約

In [ ]:
%%sql -r dataframe_10
-- データの確認
SELECT * FROM handson.car_insurance.insurance_feedback;

In [ ]:
%%sql -r dataframe_9
-- 関数実行
SELECT 
    AI_AGG(
        feedback_text, 
        'サービス全般に関する満足度について簡単に要約してください'
    ) as overall_service_insights
FROM handson.car_insurance.insurance_feedback;